In [1]:
%pip install -q langchain-community langchain-google-genai langchain-core faiss-cpu scikit-learn pandas numpy tqdm
%pip install --upgrade langchain
%pip install --upgrade langchain-text-splitters
%pip install --upgrade langchain_huggingface
%pip install --upgrade sentence-transformers

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip uninstall -y torch
%pip install torch --index-url https://download.pytorch.org/whl/cu126

Found existing installation: torch 2.14.1+cu126
Uninstalling torch-2.14.1+cu126:
  Successfully uninstalled torch-2.14.1+cu126
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu126
  Using cached torch-2.14.1%2Bcu126-cp312-cp312-win_amd64.whl.metadata (37 kB)
Using cached torch-2.14.1%2Bcu126-cp312-cp312-win_amd64.whl (2602.8 MB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install --upgrade faiss-cpu
from langchain_community.vectorstores import FAISS

Note: you may need to restart the kernel to use updated packages.


C:\Users\Duoc\AppData\Local\Temp\ipykernel_17720\414972594.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
#from google.colab import drive
#drive.mount('/content/drive')

In [3]:
import pandas as pd
import os
import zipfile

base_path = os.getcwd() 
zip_path = os.path.join(base_path, 'Dataset.zip')
extract_path = os.path.join(base_path, 'dataset')

if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(base_path)
    if os.path.exists(os.path.join(base_path, 'Dataset')):
        os.rename(os.path.join(base_path, 'Dataset'), extract_path)

res_path = os.path.join(base_path, 'RES')
SHEET_ID = "1g1qkvooaqaGk0rqgGTQw2YPltc_CZd1km-g33_vxMCE"
csv_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv"

df = pd.read_csv(csv_url)
csv_output = os.path.join(base_path, "res.csv")
df.to_csv(csv_output, index=False)


In [6]:
# Automatically detect GPU
import subprocess

try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if result.returncode == 0:
        print("CUDA / NVIDIA GPU is available and functional!")
        print(result.stdout)
        DEVICE = "cuda"
        DEVICE_INT = 0  # Use GPU 0
    else:
        print("nvidia-smi returned an error.")
        DEVICE = "cpu"
        DEVICE_INT = -1  # Use CPU
except FileNotFoundError:
    print("NVIDIA driver not detected or nvidia-smi is not in system PATH.")

CUDA / NVIDIA GPU is available and functional!
Thu Nov 26 12:05:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 621.24                 KMD Version: 621.24        CUDA UMD Version: 13.5     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650      WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   47C    P8              4W /   50W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+

In [7]:
""" import os
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

print(f"Using device: {DEVICE} (DEVICE_INT: {DEVICE_INT})")
# === Configuration ===
DOCUMENTS_PATH = "export_1"
EMBEDDING_MODEL = "keepitreal/vietnamese-sbert"
BATCH_SIZE = 1024
OUTPUT_DIR = "faiss_db"

# === Load text documents ===
def load_text_documents(path):
  documents = []
  for filename in os.listdir(path):
      if filename.endswith(".txt"):
          file_path = os.path.join(path, filename)
          loader = TextLoader(file_path, encoding='utf-8')
          raw_docs = loader.load()
          for doc in raw_docs:
              # Normalize content
              content = doc.page_content.replace("\n", " ").lower().strip()
              doc.page_content = content
              documents.append(doc)
  return documents

print("Loading text documents...")
raw_documents = load_text_documents(DOCUMENTS_PATH)
print(f"Total documents found: {len(raw_documents)}")

subset_idx = len(raw_documents)
raw_documents = raw_documents[:subset_idx]
print(f"Documents selected for embedding: {len(raw_documents)}")

# === Split documents into smaller chunks ===
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=50,
    length_function=len
)

texts = []
for doc in tqdm(raw_documents, desc="Splitting documents into chunks"):
    chunks = text_splitter.split_documents([doc])
    texts.extend(chunks)

print(f"Total chunks created: {len(texts)}")

# === Initialize embeddings ===
print(f"Loading embedding model: {EMBEDDING_MODEL}")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": DEVICE}
)

# === Build FAISS vectorstore ===
vectorstore = None

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding and adding to FAISS"):
    batch = texts[i:i + BATCH_SIZE]
    batch_embeddings = embeddings.embed_documents([t.page_content for t in batch])

    if vectorstore is None:
        vectorstore = FAISS.from_texts(
            [t.page_content for t in batch],
            embeddings
        )
    else:
        vectorstore.add_texts(
            [t.page_content for t in batch],
            embedding=batch_embeddings
        )

# === Save FAISS index ===
os.makedirs(OUTPUT_DIR, exist_ok=True)
vectorstore.save_local(OUTPUT_DIR)
print(f"FAISS vectorstore successfully saved to '{OUTPUT_DIR}'") """

' import os\nfrom tqdm import tqdm\nfrom langchain_text_splitters import RecursiveCharacterTextSplitter\nfrom langchain_community.document_loaders import TextLoader\nfrom langchain_community.vectorstores import FAISS\nfrom langchain_community.embeddings import HuggingFaceEmbeddings\n\nprint(f"Using device: {DEVICE} (DEVICE_INT: {DEVICE_INT})")\n# === Configuration ===\nDOCUMENTS_PATH = "export_1"\nEMBEDDING_MODEL = "keepitreal/vietnamese-sbert"\nBATCH_SIZE = 1024\nOUTPUT_DIR = "faiss_db"\n\n# === Load text documents ===\ndef load_text_documents(path):\n  documents = []\n  for filename in os.listdir(path):\n      if filename.endswith(".txt"):\n          file_path = os.path.join(path, filename)\n          loader = TextLoader(file_path, encoding=\'utf-8\')\n          raw_docs = loader.load()\n          for doc in raw_docs:\n              # Normalize content\n              content = doc.page_content.replace("\n", " ").lower().strip()\n              doc.page_content = content\n             

In [8]:
import os, json, shutil
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

print(f"Using device: {DEVICE} (DEVICE_INT: {DEVICE_INT})")

# === Configuration ===
DOCUMENTS_PATH = "export_1"
EMBEDDING_MODEL = "keepitreal/vietnamese-sbert"
BATCH_SIZE = 256
SAVE_EVERY = 50            # batches between checkpoints (50 x 256 = 12,800 chunks)
CHUNK_SIZE = 800           # vietnamese-sbert truncates around 256 tokens
CHUNK_OVERLAP = 80
OUTPUT_DIR = "faiss_db"
CKPT_DIR = "faiss_ckpt"

# === Load text documents ===
def load_text_documents(path):
    documents = []
    for filename in sorted(os.listdir(path)):      # sorted -> same order every run
        if filename.endswith(".txt"):
            file_path = os.path.join(path, filename)
            raw_docs = TextLoader(file_path, encoding="utf-8").load()
            for doc in raw_docs:
                doc.page_content = doc.page_content.replace("\n", " ").lower().strip()
                documents.append(doc)
    return documents

print("Loading text documents...")
raw_documents = load_text_documents(DOCUMENTS_PATH)
print(f"Total documents found: {len(raw_documents)}")

# === Split documents into chunks (one call over the whole list) ===
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
)
texts = text_splitter.split_documents(raw_documents)
print(f"Total chunks created: {len(texts)}")

# === Initialize embeddings ===
print(f"Loading embedding model: {EMBEDDING_MODEL}")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": DEVICE},
    encode_kwargs={"batch_size": 64},
)

# === Checkpoint helpers ===
def save_checkpoint(vs, next_index):
    tmp = CKPT_DIR + "_tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    vs.save_local(tmp)
    with open(os.path.join(tmp, "progress.json"), "w") as f:
        json.dump({
            "next_index": next_index,
            "total": len(texts),
            "chunk_size": CHUNK_SIZE,
            "chunk_overlap": CHUNK_OVERLAP,
            "model": EMBEDDING_MODEL,
        }, f)
    shutil.rmtree(CKPT_DIR, ignore_errors=True)
    os.replace(tmp, CKPT_DIR)      # swap in only after the new one is fully written

# === Resume from checkpoint if it matches the current settings ===
vectorstore, start = None, 0
progress_file = os.path.join(CKPT_DIR, "progress.json")
if os.path.exists(progress_file):
    with open(progress_file) as f:
        prog = json.load(f)
    same = (prog["total"] == len(texts)
            and prog["chunk_size"] == CHUNK_SIZE
            and prog["chunk_overlap"] == CHUNK_OVERLAP
            and prog["model"] == EMBEDDING_MODEL)
    if same:
        vectorstore = FAISS.load_local(CKPT_DIR, embeddings, allow_dangerous_deserialization=True)
        start = prog["next_index"]
        print(f"Resuming from chunk {start}/{len(texts)}")
    else:
        print("Checkpoint doesn't match current settings. Starting over.")

# === Build FAISS vectorstore with periodic checkpoints ===
total_batches = (len(texts) + BATCH_SIZE - 1) // BATCH_SIZE
batches_done = 0
for i in tqdm(range(start, len(texts), BATCH_SIZE),
              desc="Embedding and adding to FAISS",
              initial=start // BATCH_SIZE, total=total_batches):
    batch = texts[i:i + BATCH_SIZE]
    if vectorstore is None:
        vectorstore = FAISS.from_documents(batch, embeddings)
    else:
        vectorstore.add_documents(batch)

    batches_done += 1
    if batches_done % SAVE_EVERY == 0:
        save_checkpoint(vectorstore, i + BATCH_SIZE)

# === Save final FAISS index ===
os.makedirs(OUTPUT_DIR, exist_ok=True)
vectorstore.save_local(OUTPUT_DIR)
print(f"FAISS vectorstore successfully saved to '{OUTPUT_DIR}'")

Using device: cuda (DEVICE_INT: 0)
Loading text documents...
Total documents found: 16439
Total chunks created: 463481
Loading embedding model: keepitreal/vietnamese-sbert


d:\Github-duochh-hcmut\RAG-Assignment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12836.48it/s]


Resuming from chunk 51200/463481


Embedding and adding to FAISS: 100%|██████████| 1811/1811 [5:41:26<00:00, 12.72s/it]  


FAISS vectorstore successfully saved to 'faiss_db'


In [7]:
import numpy as np
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# === Configuration ===
EMBEDDING_MODEL = "keepitreal/vietnamese-sbert"
VECTORSTORE_PATH = "faiss_db"

# === Load FAISS vectorstore ===
print("Loading FAISS vectorstore...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": DEVICE}
)

vectorstore = FAISS.load_local(
    VECTORSTORE_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)
print("FAISS vectorstore loaded successfully.")

# === Display a sample record ===
keys = list(vectorstore.docstore._dict.keys())
if not keys:
    raise ValueError("The FAISS docstore is empty. No records found.")

first_key = keys[0]
record = vectorstore.docstore._dict[first_key]

print("\n--- First Record ---")
print(f"ID: {first_key}")
print(f"Content (truncated): {record.page_content[:200]}...")
print(f"Metadata: {record.metadata}")

# === Retrieve the corresponding vector ===
index_id = 0  # first vector
vector = vectorstore.index.reconstruct(index_id)

print("\n--- Corresponding Vector ---")
print(f"Vector size: {len(vector)}")
print(f"First 5 values: {vector[:5]}")


Loading FAISS vectorstore...


C:\Users\Duoc\AppData\Local\Temp\ipykernel_2540\638860504.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
d:\Github-duochh-hcmut\RAG-Assignment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1489.26it/s]


FAISS vectorstore loaded successfully.

--- First Record ---
ID: dd00c9a5-519f-4d6c-8cc0-1527979727c3
Content (truncated): bộ  giáo dục-bộ nội vụ  ******   việt  nam dân chủ cộng hòa  độc lập - tự do - hạnh phúc  ********   số:  002-tt-lb   hà  nội, ngày 27 tháng 01 năm 1957       thông tư liên bộ   về vấn đề cử cán bộ đi...
Metadata: {'source': 'export_1\\002-TT-LB.txt'}

--- Corresponding Vector ---
Vector size: 768
First 5 values: [ 0.32267615 -0.02751632  0.0759545   0.14563265  0.12954153]


In [12]:
import os
import re
import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import Counter
import torch

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline
from langchain_google_genai import ChatGoogleGenerativeAI
from sklearn.metrics.pairwise import cosine_similarity

# ==================== LOAD DATA ====================
df = pd.read_csv("res.csv")
df = df.head(30)
print(f"Total number of questions: {len(df)}")

# # ==================== LOAD FAISS + EMBEDDINGS ====================
# embedding_model_name = "keepitreal/vietnamese-sbert"
# embeddings = HuggingFaceEmbeddings(
#     model_name=embedding_model_name,
#     model_kwargs={"device": "cpu"}  # or "cuda"
# )

vectorstore = FAISS.load_local(
    "faiss_db",
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)

# ==================== LOAD LLM ====================
""" llm = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen1.5-1.8B", #Choose your model
    task="text-generation",
    device=DEVICE_INT,  # GPU 0
    model_kwargs={"temperature": 0, "max_length": 1024}
) """

llm = HuggingFacePipeline.from_model_id(
    #model_id="Qwen/Qwen1.5-1.8B-Chat",       # Chat variant follows instructions much better
    model_id="Qwen/Qwen1.5-1.8B",       # Chat variant follows instructions much better
    task="text-generation",
    device=DEVICE_INT,  # GPU 0
    #model_kwargs={"torch_dtype": torch.float16},      # loading: half precision to save VRAM
    pipeline_kwargs={
        "max_new_tokens": 256,
        "do_sample": False,                  # greedy decoding, same as temperature=0
        "return_full_text": False,           # return only the answer, not prompt + context
    },
)
tokenizer = llm.pipeline.tokenizer

# ==================== HELPER FUNCTIONS ====================
def normalize_text(text: str) -> str:
    text = str(text).lower().replace("\n", " ").strip()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

def rag_query(question: str, k: int = 5) -> str:
    docs = vectorstore.similarity_search(question, k=k)
    context_texts = []
    for doc in docs:
        if hasattr(doc, "page_content"):
            context_texts.append(doc.page_content)
        elif isinstance(doc, dict) and "page_content" in doc:
            context_texts.append(doc["page_content"])
        else:
            context_texts.append(str(doc))
    context = "\n".join(context_texts)

    prompt = f"""
    Bạn là một trợ lý thông minh. Trả lời câu hỏi dựa trên ngữ cảnh sau.
    Nếu không đủ thông tin, hãy nói rõ điều đó, không được tự bịa đặt.

    Ngữ cảnh:
    {context}

    Câu hỏi: {question}
    Trả lời ngắn gọn và chính xác nhất có thể:
    """

    output = llm.generate([prompt],  max_new_tokens=256)
    generated_text = output.generations[0][0].text
    print(generated_text)
    return generated_text


""" def rag_query(question: str, k: int = 5) -> str:
    docs = vectorstore.similarity_search(question.lower(), k=k)   # index was lowercased
    context = "\n\n".join(d.page_content for d in docs)

    messages = [
        {"role": "system",
         "content": "Bạn là một trợ lý thông minh. Trả lời câu hỏi dựa trên ngữ cảnh được cung cấp. "
                    "Nếu không đủ thông tin, hãy nói rõ điều đó, không được tự bịa đặt."},
        {"role": "user",
         "content": f"Ngữ cảnh:\n{context}\n\nCâu hỏi: {question}\n"
                    "Trả lời ngắn gọn và chính xác nhất có thể:"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    return llm.invoke(prompt).strip() """

def cosine_similarity_score(pred: str, true: str, emb_model) -> float:
    if not pred or not true:
        return 0.0
    pred_vec = emb_model.embed_query(pred)
    true_vec = emb_model.embed_query(true)
    return float(cosine_similarity([pred_vec], [true_vec])[0][0])

def jaccard_similarity(pred: str, true: str) -> float:
    pred_words = set(normalize_text(pred).split())
    true_words = set(normalize_text(true).split())
    return len(pred_words & true_words) / len(pred_words | true_words) if pred_words and true_words else 0.0

def token_overlap_score(pred: str, true: str) -> float:
    pred_words = normalize_text(pred).split()
    true_words = normalize_text(true).split()
    if not true_words:
        return 0.0
    return sum((Counter(pred_words) & Counter(true_words)).values()) / len(true_words)

def bleu_score_simple(pred: str, true: str) -> float:
    pred_words = normalize_text(pred).split()
    true_words = normalize_text(true).split()
    if not pred_words or not true_words:
        return 0.0
    overlap = sum((Counter(pred_words) & Counter(true_words)).values())
    precision = overlap / len(pred_words)
    bp = 1.0 if len(pred_words) >= len(true_words) else np.exp(1 - len(true_words)/len(pred_words))
    return bp * precision

def rouge_l_score(pred: str, true: str) -> float:
    pred_words = normalize_text(pred).split()
    true_words = normalize_text(true).split()
    if not pred_words or not true_words:
        return 0.0
    m, n = len(pred_words), len(true_words)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            if pred_words[i-1] == true_words[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]
    precision = lcs / m
    recall = lcs / n
    return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

def semantic_containment(pred: str, true: str) -> float:
    pred_norm = normalize_text(pred)
    true_norm = normalize_text(true)
    if not true_norm:
        return 0.0
    if true_norm in pred_norm:
        return 1.0
    pred_words = set(pred_norm.split())
    true_words = set(true_norm.split())
    return 1.0 if true_words.issubset(pred_words) else 0.0

# ==================== EVALUATION ====================
results = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating RAG"):
    question = row["Câu Hỏi"]
    ground_truth = str(row["Trả lời"]).strip()

    try:
        response = rag_query(question)
    except Exception as e:
        print(f"Error at question {idx}: {e}")
        response = ""

    metrics = {
        "Cosine_Similarity": cosine_similarity_score(response, ground_truth, embeddings),
        "Jaccard_Similarity": jaccard_similarity(response, ground_truth),
        "Token_Overlap": token_overlap_score(response, ground_truth),
        "BLEU_Score": bleu_score_simple(response, ground_truth),
        "ROUGE_L": rouge_l_score(response, ground_truth)
    }

    results.append(metrics)

results_df = pd.DataFrame(results)
mean_metrics = results_df.mean().to_dict()

print("\n=== FINAL RAG EVALUATION SUMMARY ===")
for k, v in mean_metrics.items():
    print(f"{k}: {v:.4f}")

results_df.to_csv("rag_baseline_metrics.csv", index=False)
print("Metrics saved to rag_baseline_metrics.csv")


Total number of questions: 30


Loading weights: 100%|██████████| 291/291 [00:03<00:00, 85.68it/s]
d:\Github-duochh-hcmut\RAG-Assignment\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Duoc\.cache\huggingface\hub\models--Qwen--Qwen1.5-1.8B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Evaluating RAG:   0%|          | 0/30 [00

 soft OTP là hình thức xác thực bằng cách sử dụng các thông tin được lưu trữ trong các tập tin cấu hình hệ thống, chẳng hạn như:  a) thông tin về địa chỉ, địa chỉ liên kết, địa chỉ IP, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên kết, địa chỉ IP liên


Evaluating RAG:   3%|▎         | 1/30 [03:25<1:39:29, 205.86s/it]

 - Thông báo về các quy định và quy trình của tổ chức cung cấp dịch vụ Online Banking.
     - Thông báo về các quy định và quy trình của tổ chức cung cấp dịch vụ Online Banking.
     - Thông báo về các quy định và quy trình của tổ chức cung cấp dịch vụ Online Banking.
     - Thông báo về các quy định và quy trình của tổ chức cung cấp dịch vụ Online Banking.
     - Thông báo về các quy định và quy trình của tổ chức cung cấp dịch vụ Online Banking.
     - Thông báo về các quy định và quy trình của tổ chức cung cấp dịch vụ Online Banking.
     - Thông báo về các quy định và quy trình của tổ chức cung cấp dịch vụ Online Banking.
     - Thông báo về các quy định và quy trình của tổ chức cung cấp dịch vụ Online Banking.
     - Thông báo về các quy định và quy trình của tổ chức cung cấp dịch vụ Online Banking.
     - Thông báo về các quy định và quy trình của tổ chức cung cấp dịch vụ Online Banking.
     - Thông báo về các quy định và quy trình của tổ chức cung cấp dịch vụ Online Banking.
   

Evaluating RAG:   7%|▋         | 2/30 [06:51<1:35:57, 205.62s/it]

... (49 outputs left out)

Evaluating RAG:  97%|█████████▋| 29/30 [1:42:31<03:31, 211.95s/it]

 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02/02/2025 - 02


Evaluating RAG: 100%|██████████| 30/30 [1:46:07<00:00, 212.25s/it]


=== FINAL RAG EVALUATION SUMMARY ===
Cosine_Similarity: 0.5110
Jaccard_Similarity: 0.1098
Token_Overlap: 0.1507
BLEU_Score: 0.1229
ROUGE_L: 0.1300
Metrics saved to rag_baseline_metrics.csv
